In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotx

# Load dataset
df = pd.read_csv("fm_data.csv")

# Preview
print(df.head())

# Create datetime column
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    dayfirst=True
)

# Extract hour
df['hour'] = df['datetime'].dt.hour

# Sort by datetime
df = df.sort_values('datetime').reset_index(drop=True)

# Function to classify time period
def time_period(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

# Apply function
df['period'] = df['hour'].apply(time_period)

# Day of week
df['day_of_week'] = df['datetime'].dt.day_name()

# Weekend flag
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# Final preview
print(df.head())

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Choose the most active user
user = 'Babs_05'
user_df = df[df['Username'] == user]

# Count plays per calendar day
ts = user_df.groupby(user_df['datetime'].dt.date).size()

# Convert index to DatetimeIndex so pandas treats it as a proper time series
ts.index = pd.to_datetime(ts.index)

# Ensure every calendar day is present (fills gaps with 0)
ts = ts.asfreq('D').fillna(0)

print(f"Time series length : {len(ts)} days")
print(f"Date range         : {ts.index[0].date()} to {ts.index[-1].date()}")
print(f"Min plays in a day : {int(ts.min())}")
print(f"Max plays in a day : {int(ts.max())}")
print(f"Mean plays per day : {ts.mean():.1f}")
print()
print(ts)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(ts.index, ts.values, marker='o', linewidth=1.8,
        color='steelblue', markersize=4, label='Daily plays')
ax.axhline(ts.mean(), color='tomato', linestyle='--',
           linewidth=1.2, label=f'Mean ({ts.mean():.0f} plays/day)')

ax.set_title(f'Daily Play Count — {user}  (January 2021)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Songs played')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 3: ADF Stationarity Test ─────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

def run_adf(series, label='Series'):
    """
    Run the Augmented Dickey-Fuller test and print a clear summary.
    Returns True if the series is stationary, False otherwise.
    """
    result = adfuller(series, autolag='AIC')

    adf_stat  = result[0]   # the test statistic
    p_value   = result[1]   # probability value
    n_lags    = result[2]   # number of lags used
    n_obs     = result[3]   # number of observations
    crit_vals = result[4]   # critical values at 1%, 5%, 10%

    print(f"{'─'*50}")
    print(f" ADF Test — {label}")
    print(f"{'─'*50}")
    print(f"  ADF Statistic : {adf_stat:.4f}")
    print(f"  p-value       : {p_value:.4f}")
    print(f"  Lags used     : {n_lags}")
    print(f"  Observations  : {n_obs}")
    print(f"  Critical values:")
    for key, val in crit_vals.items():
        print(f"      {key}  ->  {val:.4f}")

    if p_value < 0.05:
        print(f"\n  STATIONARY  (p={p_value:.4f} < 0.05) — no differencing needed.")
        return True
    else:
        print(f"\n  NON-STATIONARY  (p={p_value:.4f} >= 0.05) — differencing required.")
        return False


# --- Test the original series ---
is_stationary = run_adf(ts, label='Original daily play counts')

In [ ]:
# ── Step 3b: Apply differencing if the series is NOT stationary ───────────────

if not is_stationary:
    print("Applying first-order differencing...\n")
    ts_diff = ts.diff().dropna()           # y't = yt - yt-1
    is_stationary_diff = run_adf(ts_diff, label='After 1st-order differencing')

    if not is_stationary_diff:
        print("\nApplying second-order differencing...\n")
        ts_diff2 = ts_diff.diff().dropna()
        run_adf(ts_diff2, label='After 2nd-order differencing')
        ts_diff = ts_diff2  # use 2nd-order for downstream steps
else:
    print("\nNo differencing needed — the original series is already stationary.")
    ts_diff = ts  # use original for all downstream steps

In [ ]:
# ── Step 3c: Side-by-side plot — Original vs Differenced ─────────────────────

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

# Top plot: original series
axes[0].plot(ts.index, ts.values, color='steelblue',
             marker='o', markersize=4, linewidth=1.8)
axes[0].axhline(ts.mean(), color='tomato', linestyle='--',
                linewidth=1.2, label=f'Mean = {ts.mean():.0f}')
axes[0].set_title('Original Series — Daily Play Count', fontsize=13)
axes[0].set_ylabel('Songs played')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom plot: differenced series
ts_diff_plot = ts.diff().dropna()
axes[1].plot(ts_diff_plot.index, ts_diff_plot.values, color='darkorange',
             marker='o', markersize=4, linewidth=1.8)
axes[1].axhline(0, color='tomato', linestyle='--', linewidth=1.2, label='Zero line')
axes[1].set_title('After 1st-Order Differencing  (yt - yt-1)', fontsize=13)
axes[1].set_ylabel('Change in plays')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Stationarity Check — {user}', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()